# Day 6: LSTM Forecasting with PyTorch Lightning

Alright, Day 6! Today we build the second half of our forecasting engine: an LSTM Neural Network.

Unlike Prophet, which is a statistical model, an LSTM is a type of Recurrent Neural Network (RNN) designed specifically to remember long-term patterns in sequential data. 

**Today's Agenda:**
1. **Data Prep for Deep Learning:** LSTMs require data to be scaled (0 to 1) and structured into "sliding windows" (sequences).
2. **PyTorch Datasets:** Create custom data loaders to feed batches of data to the GPU/CPU.
3. **PyTorch Lightning Model:** Build a clean, modular LSTM architecture.
4. **Training & Evaluation:** Train the model and compare its MAPE against our Prophet baseline.
5. **MLflow Tracking:** Log the deep learning metrics to our experiment tracker.

Let's get to work.

In [1]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import mlflow
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Load the prepared time-series data from Day 4
print("Loading prepared time-series data...")
daily_sales = pd.read_csv('data/daily_sales_prepared.csv', parse_dates=['date'])

# We only need the target variable for the LSTM
sales_data = daily_sales[['total_amount']].values
print(f"Loaded {len(sales_data)} days of data.")

Loading prepared time-series data...
Loaded 541 days of data.


### Step 1: Scaling and Windowing (Crucial for LSTMs)

Neural networks are incredibly sensitive to the scale of input data. If we feed them raw dollar amounts (like $5,000), the gradients will explode. We must scale everything between 0 and 1 using `MinMaxScaler`.

Furthermore, LSTMs don't understand "dates". They understand **sequences**. We need to convert our continuous timeline into "sliding windows". 
*Rule:* We will use the past **30 days** of sales to predict the **next 1 day**.

In [2]:
# 1. Scale the data between 0 and 1
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(sales_data)

# 2. Train/Test Split (Same 30-day test set as Prophet for fair comparison)
test_days = 30
train_data = scaled_data[:-test_days]
test_data = scaled_data[-test_days:]

# 3. Function to create sliding window sequences
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]       # Past 30 days
        y = data[i + seq_length]           # The next day
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 30

# Create training sequences
X_train, y_train = create_sequences(train_data, SEQ_LENGTH)

# Create test sequences (we need the last 30 days of TRAINING to predict the first day of TESTING)
full_data = scaled_data
X_test, y_test = create_sequences(full_data[-(test_days + SEQ_LENGTH):], SEQ_LENGTH)

print(f"Training sequences shape: {X_train.shape} (Samples, 30 Days, 1 Feature)")
print(f"Testing sequences shape: {X_test.shape}")

Training sequences shape: (481, 30, 1) (Samples, 30 Days, 1 Feature)
Testing sequences shape: (30, 30, 1)


### Step 2: PyTorch Datasets and DataLoaders

In PyTorch, we don't just pass raw arrays to the model. We wrap them in a `Dataset` class, and then use a `DataLoader` to feed them to the model in "batches" during training. This makes the training process highly efficient.

In [3]:
# Custom PyTorch Dataset class
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        # Convert numpy arrays to PyTorch tensors (float32 is required for LSTMs)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = TimeSeriesDataset(X_train, y_train)
test_dataset = TimeSeriesDataset(X_test, y_test)

# Create DataLoaders (Batch size of 32 is standard)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("✅ PyTorch DataLoaders created successfully!")

✅ PyTorch DataLoaders created successfully!


### Step 3: Building the PyTorch Lightning Model

This is where the magic happens. Instead of writing hundreds of lines of standard PyTorch training loops, **PyTorch Lightning** lets us define the model architecture, the loss function, and the optimizer in one clean, organized class.

Our architecture:
- **LSTM Layer:** 2 layers, 64 hidden units (to capture complex patterns).
- **Linear Layer:** Takes the final output of the LSTM and squashes it down to 1 number (the predicted sales).

In [4]:
class LSTMForecaster(pl.LightningModule):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        # The LSTM layer (batch_first=True means our data shape is [Batch, Seq_Length, Features])
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout)
        
        # The final output layer (predicting 1 value)
        self.fc = nn.Linear(hidden_size, 1)
        
        # Loss function (Mean Squared Error is standard for regression)
        self.criterion = nn.MSELoss()

    def forward(self, x):
        # Pass data through LSTM
        lstm_out, _ = self.lstm(x)
        # Take only the output from the very last time step (the 30th day)
        out = self.fc(lstm_out[:, -1, :]) 
        return out

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('val_loss', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        # Adam optimizer is the industry standard for LSTMs
        return torch.optim.Adam(self.parameters(), lr=0.001)

print("✅ LSTM Model Architecture defined!")

✅ LSTM Model Architecture defined!


### Step 4: Training the Model

Now we initialize the PyTorch `Trainer`. We will train for 30 epochs (passes through the entire dataset). 
*Note: If you have an NVIDIA GPU, PyTorch Lightning will automatically detect it. Since we are on a standard laptop, we will force it to use the CPU.*

In [5]:
# Initialize the model
model = LSTMForecaster()

# Initialize the PyTorch Lightning Trainer
trainer = pl.Trainer(
    max_epochs=30,           # Train for 30 full passes
    accelerator='cpu',       # Use CPU (change to 'gpu' if you have CUDA)
    enable_progress_bar=True,
    enable_model_summary=True,
    log_every_n_steps=5
)

print("🚀 Starting LSTM Training...")
# Train the model (we pass test_loader as the val_loader to monitor validation loss)
trainer.fit(model, train_loader, test_loader)
print("✅ Training Complete!")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name      | Type    | Params | Mode  | FLOPs
------------------------------------------------------
0 | lstm      | LSTM    | 50.4 K | train | 0    
1 | fc        | Linear  | 65     | train | 0    
2 | criterion | MSELoss | 0      | train | 0    
------------------------------------------------------
50.5 K    Trainable params
0         Non-trainable params
50.5 K    Total params
0.202     Total estimated model params size (MB)
3         Modules in train mode
0         Modules in 

🚀 Starting LSTM Training...


Sanity Checking: |                          | 0/? [00:00<?, ?it/s]

Training: |                                 | 0/? [00:00<?, ?it/s]

Validation: |                               | 0/? [00:00<?, ?it/s]

Validation: |                               | 0/? [00:00<?, ?it/s]

... (26 outputs left out)

Validation: |                               | 0/? [00:00<?, ?it/s]

Validation: |                               | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.


✅ Training Complete!


### Step 5: Evaluation & Comparison with Prophet

Let's see how our Deep Learning model performed on the exact same 30-day test set we used for Prophet. We need to inverse-transform our predictions back to real dollar amounts to calculate the MAPE.

In [6]:
# Set model to evaluation mode
model.eval()
predictions_scaled = []

# Generate predictions without calculating gradients (saves memory)
with torch.no_grad():
    for x, _ in test_loader:
        pred = model(x)
        predictions_scaled.append(pred.numpy())

# Flatten and inverse transform back to dollars
predictions_scaled = np.concatenate(predictions_scaled)
predictions_inv = scaler.inverse_transform(predictions_scaled)
actuals_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate Metrics
mape_lstm = mean_absolute_percentage_error(actuals_inv, predictions_inv) * 100
rmse_lstm = np.sqrt(mean_squared_error(actuals_inv, predictions_inv))

print("--- LSTM Model Evaluation ---")
print(f"LSTM MAPE: {mape_lstm:.2f}%")
print(f"LSTM RMSE: ${rmse_lstm:.2f}")

--- LSTM Model Evaluation ---
LSTM MAPE: 18.81%
LSTM RMSE: $8132.25


### Step 6: Logging to MLflow

We must log this to MLflow to maintain our MLOps tracking. We will log it under the exact same experiment we created yesterday.

In [7]:
# Set the experiment name (same as Day 5)
mlflow.set_experiment("RetailPulse_Demand_Forecasting")

with mlflow.start_run(run_name="Baseline_LSTM_v1"):
    
    # 1. Log Parameters
    mlflow.log_param("model_type", "LSTM_PyTorch")
    mlflow.log_param("sequence_length", SEQ_LENGTH)
    mlflow.log_param("hidden_size", 64)
    mlflow.log_param("num_layers", 2)
    mlflow.log_param("epochs", 30)
    
    # 2. Log Metrics
    mlflow.log_metric("mape_percent", round(mape_lstm, 2))
    mlflow.log_metric("rmse", round(rmse_lstm, 2))
    
    # 3. Create and Log Visualization
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(daily_sales['date'].iloc[-test_days:], actuals_inv, label='Actual Sales', color='blue', marker='o')
    ax.plot(daily_sales['date'].iloc[-test_days:], predictions_inv, label='Predicted Sales (LSTM)', color='orange', marker='x')
    ax.set_title('LSTM vs Prophet: Actual vs Predicted (Last 30 Days)')
    ax.set_xlabel('Date')
    ax.set_ylabel('Total Sales ($)')
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    plt.savefig("lstm_baseline_eval.png")
    mlflow.log_artifact("lstm_baseline_eval.png")
    plt.close()
    
    print("✅ Logged LSTM parameters, metrics, and plot to MLflow!")

✅ Logged LSTM parameters, metrics, and plot to MLflow!


### Wrap Up

Incredible work! You just built, trained, and evaluated a Deep Learning time-series model using PyTorch Lightning. 

If you check your MLflow UI (`mlflow ui`), you will now see **two** runs under the "RetailPulse_Demand_Forecasting" experiment:
1. `Baseline_Prophet_v1`
2. `Baseline_LSTM_v1`

This is exactly how MLOps works in the real world. We don't just pick one model; we track them all and compare. 

Tomorrow (Day 7) is our Week 1 Checkpoint. We will review our EDA, ensure our data is perfect, and prepare to build the **Hybrid Ensemble Model** that combines Prophet and LSTM to crush that 12% MAPE target!